Download the md files from llm-zoomcamp course github repo

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents


In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DatatalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter= lambda path: "/lessons" in path
)
files = reader.read()
print("number of md files = ",len(files))
documents =[]
for file in files:
    doc = file.parse()
    documents.append(doc)
len(documents)
documents[7]

number of md files =  72


{'content': '# RAG Helper\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=JxaC6Hrym6c&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lessons, we built the RAG flow piece by piece -\nsearch, then the prompt, then the LLM call. The pipeline works, but\nevery time we want to use it, we need to repeat the same code.\n\nWe\'ll use this code throughout the course, so let\'s put it into two\nreusable files:\n\n- [ingest.py](../code/ingest.py) - loading data and building the search index\n- [rag_helper.py](../code/rag_helper.py) - the RAG logic (search, prompt, LLM)\n\nThen in notebooks, we just import from these files and use them.\n\n## ingest.py\n\nThis file handles data loading and index creation - everything we\nneed before we can search.\n\nCreate `ingest.py` with two functions:\n\n```python\nimport requests\nfrom minsearch import Index\n\ndef load_faq_data():\n    docs_url = "https://datatalks.club/faq/json/courses.json"\n    response = requests.get(docs_url)\

In [ ]:
from minsearch import Index
index = Index(
    text_fields=["content","filename"]
    
)
chunks = chunk_documents(documents,size=2000,step=1000)
print("Number of chunks= ",len(chunks))

index.fit(chunks)
question ="How does the agentic loop keep calling the model until it stops?"
index.search(question)
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
client = OpenAI()
from rag_helper import RAGBase
rag_base = RAGBase(index,client)
rag_base.rag(question)


Number of chunks=  295
searchfunction called
tokens used:  2491


'The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether any `function_call` items were returned:\n\n- If there is at least one function call, it runs the tool, appends the result, and loops again.\n- If there are no function calls in that turn, it breaks out of the loop.\n\nSo the stopping condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nThat means the agent keeps going until the model returns a final message with no more tool calls.'

In [5]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


In [15]:
agent_tools = Tools()
def search(query:str)->dict[str,str]:
        """search the llm-zoomcamp course using the given query"""
        print("searchfunction called")
        boost_dict = {'content': 3.0, 'filename': 0.5}

        return index.search(
            query,
            num_results=5,
            boost_dict=boost_dict
        )


agent_tools.add_tool(search)
agent_tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'search the llm-zoomcamp course using the given query',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [16]:


chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)



In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using context, don't do it yourself. Only use the 
facts from the context.

At the end, ask if there are other areas that the user wants to explore.
"""

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)



In [19]:


result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)



-> Response received
searchfunction called


-> Response received
searchfunction called


-> Response received
searchfunction called


-> Response received
